In [ ]:
import pandas as pd
from glob import glob
from analysis.processors.base import BaseProcessor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea.processor import run_uproot_job, iterative_executor, futures_executor
NanoAODSchema.warn_missing_crossrefs = False


workflow = "ztomumu"
year = "2016preVFP"
dataset = "DYJetsToLL_50_1"
output_format = "coffea"
executor = "futures"

if year.startswith("201"):
    nano_version = "9"
else:
    nano_version = "12"


fileset = {
    "2016preVFP": {
        "DYJetsToLL_50_1": ["root://xcache///store/mc/RunIISummer20UL16NanoAODAPVv9/DYJetsToLL_M-50_HT-70to100_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_preVFP_v11-v2/230000/0FEEC8CC-15E4-094E-B554-440C76A25697.root"],
        "SingleMuonBv2_1": ["root://xcache///store/data/Run2016B/SingleMuon/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v2/2520000/0BDA2469-4879-A343-9F6D-2F05696FFC2A.root", "root://xcache///store/data/Run2016B/SingleMuon/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v2/2520000/30536D41-5D52-0640-A3F0-CF7C44FE4AB6.root"],
    },
    "2016postVFP": {
        "DYJetsToLL_50_1": ["root://xcache///store/mc/RunIISummer20UL16NanoAODAPVv9/DYJetsToLL_M-50_HT-70to100_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_preVFP_v11-v2/230000/0FEEC8CC-15E4-094E-B554-440C76A25697.root"],
        "SingleMuonG_1": ["root://xcache///store/data/Run2016G/SingleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/130000/0A4230E2-0C75-604D-890F-A4CE5E5C164E.root"] ,
    },
    "2017": {
        "DYJetsToLL_50_1": ["root://xcache//store/mc/RunIISummer20UL17NanoAODv9/DYJetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_mc2017_realistic_v9-v1/270000/FD918196-E5D8-5444-A397-9EF7E3458FBC.root",],
        "TTTo2L2Nu_1": ["root://xcache//store/mc/RunIISummer20UL17NanoAODv9/TTTo2L2Nu_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_mc2017_realistic_v9-v1/2510000/011F120E-2342-D046-A22B-88307336493B.root"],
        "SingleMuonB_1": ["root://xcache//store/data/Run2017B/SingleMuon/NANOAOD/UL2017_MiniAODv2_NanoAODv9-v1/120000/09FD9FD6-A164-9A45-80BB-F3D1FBF9C462.root"],
    },
    "2018": {
        "DYJetsToLL_50_1": ["root://xcache//store/mc/RunIISummer20UL18NanoAODv2/DYJetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v15_L1v1-v1/270000/0262164B-9CA8-8F44-A9B9-2E0056FD9428.root"],
        "SingleMuonA_1": ["root://xcache//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/00EBBD1F-032C-9B49-A998-7645C9966432.root", "root://xcache//store/data/Run2018A/SingleMuon/NANOAOD/UL2018_MiniAODv2_NanoAODv9-v2/2550000/28FF17A8-95EB-FD41-A55B-2EFAF2D6AF91.root"]
    },
    "2022preEE": {
        "DYJetsToLL_50_1": ["root://xcache//store/mc/Run3Summer22NanoAODv12/DYto2L-2Jets_MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/2520000/181836d0-879c-4c05-93c1-69956def2efb.root"],
        "ZH": ["root://xcache//store/mc/Run3Summer22NanoAODv12/ZHto2Zto4L_M125_TuneCP5_13p6TeV_powheg2-minlo-HZJ-JHUGenV752-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/2520000/05ca677b-232e-4630-80c7-6ae0bb698a45.root"],
    },
    "2022postEE": {
        "DYJetsToLL_50_1": ["root://xcache//store/mc/Run3Summer22EENanoAODv12/DYto2L-2Jets_MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_postEE_v6-v2/2520000/098581d9-40df-4e56-9e33-f5d452fa4ee3.root"],
    },
    "2023preBPix": {
        "DYJetsToLL_50_1": ["root://xcache//store/mc/Run3Summer23NanoAODv12/DYto2L-2Jets_MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/NANOAODSIM/130X_mcRun3_2023_realistic_v14-v1/2550000/00da63ff-7a11-4939-b7d8-96610222c5c6.root"],
        "ZHto2Zto4L": ["root://xcache//store/mc/Run3Summer23NanoAODv12/ZH_Hto2Z_4LFilter_M-125_TuneCP5_13p6TeV_powheg-jhugenv752-pythia8/NANOAODSIM/130X_mcRun3_2023_realistic_v15-v3/50000/084aa81e-8fdd-420e-b69f-40b039b6c9d9.root"],
    },
    "2023postBPix": {
        "DYJetsToLL_50_1": ["root://xcache//store/mc/Run3Summer23BPixNanoAODv12/DYto2L-2Jets_MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/NANOAODSIM/130X_mcRun3_2023_realistic_postBPix_v2-v3/80000/03abf7f7-3b4e-4692-aa6d-e193a5a9760b.root"],
        "ZHto2Zto4L": ["root://xcache//store/mc/Run3Summer23BPixNanoAODv12/ZH_Hto2Z_4LFilter_M-125_TuneCP5_13p6TeV_powheg-jhugenv752-pythia8/NANOAODSIM/130X_mcRun3_2023_realistic_postBPix_v6-v2/40000/05215ce3-a3f1-4a04-9e96-cc4875485e28.root"],
    },
}

out = run_uproot_job(
    {dataset: fileset[year][dataset]},
    treename="Events",
    processor_instance=BaseProcessor(
        workflow=workflow, 
        year=year,
        nano_version=nano_version,
        output_format=output_format,
        output_location="test/"
    ),
    executor=iterative_executor if executor == "iterative" else futures_executor,
    executor_args={
        "schema": NanoAODSchema,
        "workers": 1 if executor == "iterative" else 4,
    }
)

In [ ]:
out["metadata"]

In [ ]:
out["histograms"]